# SETUP — Pre-poblar artefactos que requieren Python
**Ejecutar UNA VEZ** antes del HOL. Crea:
1. Modelo Fraude V1 (champion) + V2 (challenger)
2. Modelo Churn V1 (champion)
3. Model Monitor (observabilidad)
4. Múltiples ejecuciones de inference para historial

**NO es para el participante** — solo para setup de la cuenta.

In [ ]:
USE ROLE ACCOUNTADMIN;
USE DATABASE CREDIBANCO_HOL;
USE WAREHOUSE CREDIBANCO_HOL_WH;
USE SCHEMA ANALITICA;

## 1. Modelo Fraude: V1 (champion) + V2 (challenger)

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

session = get_active_session()
reg = Registry(session, database_name='CREDIBANCO_HOL', schema_name='ANALITICA')

# Datos fraude
train_pd = session.table('CREDIBANCO_HOL.ANALITICA.TRAIN_RIESGO_COMERCIO').to_pandas()
feature_cols = [c for c in train_pd.columns if c not in ['COMERCIO_ID', 'ES_FRAUDE']]
X = train_pd[feature_cols].fillna(0)
y = train_pd['ES_FRAUDE']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

neg, pos = int((y_train == 0).sum()), int((y_train == 1).sum())
spw = neg / pos if pos > 0 else 1

# V1
model_v1 = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, scale_pos_weight=spw, random_state=42, eval_metric='logloss')
model_v1.fit(X_train, y_train)
y_pred = model_v1.predict(X_test); y_proba = model_v1.predict_proba(X_test)[:, 1]
m1 = {'accuracy': accuracy_score(y_test, y_pred), 'precision': precision_score(y_test, y_pred, zero_division=0), 'recall': recall_score(y_test, y_pred, zero_division=0), 'f1': f1_score(y_test, y_pred, zero_division=0), 'auc_roc': roc_auc_score(y_test, y_proba)}
try: reg.delete_model('MODELO_FRAUDE_CREDIBANCO')
except: pass
mv1 = reg.log_model(model_v1, model_name='MODELO_FRAUDE_CREDIBANCO', version_name='V1', sample_input_data=X_train.head(10), metrics=m1, conda_dependencies=['xgboost','scikit-learn'], target_platforms=['WAREHOUSE'], comment='XGBoost fraude V1')
mv1.set_alias('champion')
print(f'V1: {m1}')

# V2
model_v2 = XGBClassifier(n_estimators=200, max_depth=7, learning_rate=0.05, scale_pos_weight=spw, random_state=42, eval_metric='logloss')
model_v2.fit(X_train, y_train)
y_pred2 = model_v2.predict(X_test); y_proba2 = model_v2.predict_proba(X_test)[:, 1]
m2 = {'accuracy': accuracy_score(y_test, y_pred2), 'precision': precision_score(y_test, y_pred2, zero_division=0), 'recall': recall_score(y_test, y_pred2, zero_division=0), 'f1': f1_score(y_test, y_pred2, zero_division=0), 'auc_roc': roc_auc_score(y_test, y_proba2)}
mv2 = reg.log_model(model_v2, model_name='MODELO_FRAUDE_CREDIBANCO', version_name='V2', sample_input_data=X_train.head(10), metrics=m2, conda_dependencies=['xgboost','scikit-learn'], target_platforms=['WAREHOUSE'], comment='XGBoost fraude V2 tuned')
mv2.set_alias('challenger')
print(f'V2: {m2}')

## 2. Modelo Churn: V1 (champion)

In [ ]:
churn_pd = session.table('CREDIBANCO_HOL.ANALITICA.COMERCIOS_CHURN').to_pandas()
fc = [c for c in churn_pd.columns if c not in ['COMERCIO_ID', 'ES_CHURN']]
Xc = churn_pd[fc].fillna(0); yc = churn_pd['ES_CHURN']
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)
neg_c, pos_c = int((yc_train==0).sum()), int((yc_train==1).sum())
spw_c = neg_c / pos_c if pos_c > 0 else 1
model_churn = XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.08, scale_pos_weight=spw_c, random_state=42, eval_metric='logloss')
model_churn.fit(Xc_train, yc_train)
yc_pred = model_churn.predict(Xc_test); yc_proba = model_churn.predict_proba(Xc_test)[:, 1]
mc = {'accuracy': accuracy_score(yc_test, yc_pred), 'precision': precision_score(yc_test, yc_pred, zero_division=0), 'recall': recall_score(yc_test, yc_pred, zero_division=0), 'f1': f1_score(yc_test, yc_pred, zero_division=0), 'auc_roc': roc_auc_score(yc_test, yc_proba)}
try: reg.delete_model('MODELO_CHURN_COMERCIOS')
except: pass
mv_ch = reg.log_model(model_churn, model_name='MODELO_CHURN_COMERCIOS', version_name='V1', sample_input_data=Xc_train.head(10), metrics=mc, conda_dependencies=['xgboost','scikit-learn'], target_platforms=['WAREHOUSE'], comment='XGBoost churn comercios')
mv_ch.set_alias('champion')
print(f'Churn V1: {mc}')

## 3. Ejecutar Inference múltiples veces (generar historial)

In [ ]:
# Run inference with both models to generate usage history
import snowflake.snowpark.functions as F

# Fraud model inference
fraud_champion = reg.get_model('MODELO_FRAUDE_CREDIBANCO').version('V1')
scoring_df = session.table('CREDIBANCO_HOL.ANALITICA.TRAIN_RIESGO_COMERCIO')
scoring_cols = [c for c in scoring_df.columns if c not in ['COMERCIO_ID', 'ES_FRAUDE']]
input_df = scoring_df.select(scoring_cols).limit(500)

# Run 3 times to generate history
for i in range(3):
    preds = fraud_champion.run(input_df, function_name='predict')
    count = preds.count()
    print(f'Fraud inference run {i+1}: {count} predictions')

# Churn model inference
churn_champion = reg.get_model('MODELO_CHURN_COMERCIOS').version('V1')
churn_df = session.table('CREDIBANCO_HOL.ANALITICA.COMERCIOS_CHURN')
churn_cols = [c for c in churn_df.columns if c not in ['COMERCIO_ID', 'ES_CHURN']]
churn_input = churn_df.select(churn_cols).limit(500)

for i in range(3):
    preds_ch = churn_champion.run(churn_input, function_name='predict')
    count_ch = preds_ch.count()
    print(f'Churn inference run {i+1}: {count_ch} predictions')

# Also run V2 challenger
fraud_v2 = reg.get_model('MODELO_FRAUDE_CREDIBANCO').version('V2')
preds_v2 = fraud_v2.run(input_df, function_name='predict')
print(f'Fraud V2 inference: {preds_v2.count()} predictions')
print('Historial de inference generado')

## 4. Verificación

In [ ]:
print('=== MODELOS REGISTRADOS ===')
session.sql('SHOW MODELS IN SCHEMA CREDIBANCO_HOL.ANALITICA').show()
print('\n=== SETUP COMPLETADO ===')